#Download the Dataset via Drive

In [23]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#Step 1: Load and Inspect the Raw Data

In [24]:
import pandas as pd
import numpy as np

In [25]:
#Load and handle '?' values
df = pd.read_csv('/content/drive/MyDrive/AI RESEARCH/SPE NAICE/household_power_consumption.txt',
                 sep=';',
                 na_values=['?'],
                 low_memory=False
                 )
df.head()

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


#Step 2: Create Datetime Index

In [26]:
df['dt'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], dayfirst=True)
df.set_index('dt', inplace=True)
df.drop(['Date', 'Time'], axis=1, inplace=True)
df.head()

,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
dt,,,,,,,
2006-12-16 17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
2006-12-16 17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2006-12-16 17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
2006-12-16 17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
2006-12-16 17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


In [27]:
print(df.index)

DatetimeIndex(['2006-12-16 17:24:00', '2006-12-16 17:25:00',
               '2006-12-16 17:26:00', '2006-12-16 17:27:00',
               '2006-12-16 17:28:00', '2006-12-16 17:29:00',
               '2006-12-16 17:30:00', '2006-12-16 17:31:00',
               '2006-12-16 17:32:00', '2006-12-16 17:33:00',
               ...
               '2010-11-26 20:53:00', '2010-11-26 20:54:00',
               '2010-11-26 20:55:00', '2010-11-26 20:56:00',
               '2010-11-26 20:57:00', '2010-11-26 20:58:00',
               '2010-11-26 20:59:00', '2010-11-26 21:00:00',
               '2010-11-26 21:01:00', '2010-11-26 21:02:00'],
              dtype='datetime64[ns]', name='dt', length=2075259, freq=None)


#Step 3: Handle Missing Values

In [28]:
print(df.shape)
print(df.isnull().sum())

(2075259, 7)
Global_active_power      25979
Global_reactive_power    25979
Voltage                  25979
Global_intensity         25979
Sub_metering_1           25979
Sub_metering_2           25979
Sub_metering_3           25979
dtype: int64


In [29]:
#Fill missing values using forward fill
df.ffill(inplace=True)

#Confirm no more missing values
print(df.isnull().sum())

Global_active_power      0
Global_reactive_power    0
Voltage                  0
Global_intensity         0
Sub_metering_1           0
Sub_metering_2           0
Sub_metering_3           0
dtype: int64


#Step 4: Resample from Minute → Hourly

Since the data is recorded every minute, we need to aggregate it to hourly intervals for our prediction task.

In [30]:
#Resample to hourly by taking the mean
df_hourly = df.resample('H').mean()

print(df_hourly.shape)
print(df_hourly.head())

(34589, 7)
                     Global_active_power  Global_reactive_power     Voltage  \
dt                                                                            
2006-12-16 17:00:00             4.222889               0.229000  234.643889   
2006-12-16 18:00:00             3.632200               0.080033  234.580167   
2006-12-16 19:00:00             3.400233               0.085233  233.232500   
2006-12-16 20:00:00             3.268567               0.075100  234.071500   
2006-12-16 21:00:00             3.056467               0.076667  237.158667   

                     Global_intensity  Sub_metering_1  Sub_metering_2  \
dt                                                                      
2006-12-16 17:00:00         18.100000             0.0        0.527778   
2006-12-16 18:00:00         15.600000             0.0        6.716667   
2006-12-16 19:00:00         14.503333             0.0        1.433333   
2006-12-16 20:00:00         13.916667             0.0        0.000000 

/tmp/ipykernel_283/1960342433.py:2: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_hourly = df.resample('H').mean()


#Step 5: Engineer Time-Based Features
Now we add temporal features that help the model understand patterns like time of day, day of week, weekends etc.

In [31]:
df_hourly['hour']        = df_hourly.index.hour
df_hourly['day_of_week'] = df_hourly.index.dayofweek
df_hourly['month']       = df_hourly.index.month
df_hourly['is_weekend']  = df_hourly['day_of_week'].isin([5, 6]).astype(int)
df_hourly['is_daytime']  = df_hourly['hour'].between(6, 18).astype(int)

print(df_hourly[['hour', 'day_of_week', 'month', 'is_weekend', 'is_daytime']].head(10))

                     hour  day_of_week  month  is_weekend  is_daytime
dt                                                                   
2006-12-16 17:00:00    17            5     12           1           1
2006-12-16 18:00:00    18            5     12           1           1
2006-12-16 19:00:00    19            5     12           1           0
2006-12-16 20:00:00    20            5     12           1           0
2006-12-16 21:00:00    21            5     12           1           0
2006-12-16 22:00:00    22            5     12           1           0
2006-12-16 23:00:00    23            5     12           1           0
2006-12-17 00:00:00     0            6     12           1           0
2006-12-17 01:00:00     1            6     12           1           0
2006-12-17 02:00:00     2            6     12           1           0


#Step 6: Add Nigeria-Specific Power Source Features
We now simulate the 3 power sources — Grid, Solar and Generator — based on realistic Nigerian power supply patterns.

In [32]:
# Solar: available between 7am and 6pm
df_hourly['solar_available'] = df_hourly['hour'].between(7, 18).astype(int)

# Grid: simulate NEPA availability (~8 hrs/day, typically morning and evening)
np.random.seed(42)
grid_hours = [6, 7, 8, 9, 18, 19, 20, 21]
df_hourly['grid_available'] = df_hourly['hour'].isin(grid_hours).astype(int)

# Generator: kicks in when both grid and solar are unavailable
df_hourly['gen_active'] = (
    (df_hourly['grid_available'] == 0) &
    (df_hourly['solar_available'] == 0)
).astype(int)

print(df_hourly[['hour', 'grid_available', 'solar_available', 'gen_active']].head(24))

                     hour  grid_available  solar_available  gen_active
dt                                                                    
2006-12-16 17:00:00    17               0                1           0
2006-12-16 18:00:00    18               1                1           0
2006-12-16 19:00:00    19               1                0           0
2006-12-16 20:00:00    20               1                0           0
2006-12-16 21:00:00    21               1                0           0
2006-12-16 22:00:00    22               0                0           1
2006-12-16 23:00:00    23               0                0           1
2006-12-17 00:00:00     0               0                0           1
2006-12-17 01:00:00     1               0                0           1
2006-12-17 02:00:00     2               0                0           1
2006-12-17 03:00:00     3               0                0           1
2006-12-17 04:00:00     4               0                0           1
2006-1

#Step 7: Add Lag Features
Lag features tell the model what consumption looked like in the past — which is critical for time series forecasting.

In [33]:
df_hourly['lag_1h']   = df_hourly['Global_active_power'].shift(1)
df_hourly['lag_24h']  = df_hourly['Global_active_power'].shift(24)
df_hourly['lag_168h'] = df_hourly['Global_active_power'].shift(168)

print(df_hourly[['Global_active_power', 'lag_1h', 'lag_24h', 'lag_168h']].head(10))
print("\nMissing values after lag:\n", df_hourly[['lag_1h', 'lag_24h', 'lag_168h']].isnull().sum())

                     Global_active_power    lag_1h  lag_24h  lag_168h
dt                                                                   
2006-12-16 17:00:00             4.222889       NaN      NaN       NaN
2006-12-16 18:00:00             3.632200  4.222889      NaN       NaN
2006-12-16 19:00:00             3.400233  3.632200      NaN       NaN
2006-12-16 20:00:00             3.268567  3.400233      NaN       NaN
2006-12-16 21:00:00             3.056467  3.268567      NaN       NaN
2006-12-16 22:00:00             2.200133  3.056467      NaN       NaN
2006-12-16 23:00:00             2.061600  2.200133      NaN       NaN
2006-12-17 00:00:00             1.882467  2.061600      NaN       NaN
2006-12-17 01:00:00             3.349400  1.882467      NaN       NaN
2006-12-17 02:00:00             1.587267  3.349400      NaN       NaN

Missing values after lag:
 lag_1h        1
lag_24h      24
lag_168h    168
dtype: int64
